In [3]:
import ast

s = '''
match i:
    case Enum.m(): 
        return "One" 
'''

print(ast.dump(ast.parse(s).body[0].cases[0].pattern, indent=2))

MatchClass(
  cls=Attribute(
    value=Name(id='Enum', ctx=Load()),
    attr='m',
    ctx=Load()))


In [ ]:
from dataclasses import dataclass

def fun():
    return 1

@dataclass
class Point2D:
    x: int
    y: int

@dataclass
class Point3D:
    x: int
    y: int
    z: int

def describe_point(point):
    ty = 10000
    match point:
        case Point2D(Point3D(_,_), ty):
            return f"Origin in 2D {ty}"
        case Point3D(x=0, y=0, z=0):
            return "Origin in 3D"
        case Point2D([1 | 2]):
            return f"2D point at ({x}, {y})"
        case Point3D(x, y, z):
            return f"3D point at ({x}, {y}, {z})"
        case _:
            return "Unknown point"

# Example usage:
print(describe_point(Point2D(Point3D(1,2,3), 0)))      # Output: Origin in 2D
print(describe_point(Point3D(0, 0, 0)))   # Output: Origin in 3D
print(describe_point(Point2D(1, 2)))      # Output: 2D point at (1, 2)
print(describe_point(Point3D(1, 2, 3)))   # Output: 3D point at (1, 2, 3)

In [ ]:
from guppylang import guppy
from guppylang.std.quantum import qubit, array

import guppylang
from typing import Generic


guppylang.enable_experimental_features()

T = guppy.type_var("T")

@guppy.enum
class Direction(Generic[T]):
    North = {"q": T}
    South = {"q": int}
    East = {"q": int}
    West = {"q": int}


@guppy.struct
class Point():
    A: int
    B: int

    @guppy
    def method(self) -> int:
        return 23

@guppy
def main() -> int:
    a = Direction.North(3)
    match a:
        case Direction.North(2):
            return 1
        case _:
            return 4

print('....')
main.check()

In [1]:
from typing import Generic

from guppylang import guppy
import guppylang

guppylang.enable_experimental_features()

T = guppy.type_var("T")
x = "1"
@guppy.struct
class Direction(Generic[T]):
    A: T
    B: int



@guppy
def main(i: int, g: int, r: int) -> int:
    d = Direction(1, 2)
    match d:
        case Direction(1, 2):
            return 1
        case Direction(1, _):
            return 2
        case _:
            return 3


main.check()


ImportError: cannot import name 'check_call' from partially initialized module 'guppylang_internals.checker.expr_checker' (most likely due to a circular import) (/Users/nicola.assolini/Documents/GitHub/guppylang/guppylang-internals/src/guppylang_internals/checker/expr_checker.py)

In [ ]:
import ast
s = "Direction.North(2)"
print(ast.dump(ast.parse(s), indent=2))

In [ ]:
import ast
s = "3"
print(ast.dump(ast.parse(s).body[0].value, indent=2))
v = ast.parse(s).body[0].value
v.type = None

In [1]:
from guppylang import guppy
from guppylang.std.quantum import qubit, measure

@guppy.struct
class Point:
    x: int
    y: int


@guppy
def main(p: Point) -> None:
    q = qubit()  # ERROR if qubit is linear and cannot be reused
    match p:
        case Point(3, 4):
            measure(q)  # ERROR if qubit is linear and cannot be reused
        case Point(5, 6):
            b = p.x  # ERROR if p.x is linear and cannot be reused

main.check()

Error: Drop violation (at <In[1]>:12:4)
   | 
10 | @guppy
11 | def main(p: Point) -> None:
12 |     q = qubit()  # ERROR if qubit is linear and cannot be reused
   |     ^ Variable `q` with non-droppable type `qubit` is leaked

Help: Make sure that `q` is consumed or returned to avoid the leak

Guppy compilation failed due to 1 previous error


In [1]:
# Example: linear variable used in multiple match arms (should error if not allowed)
from guppylang import guppy
from guppylang.std.quantum import qubit, measure

@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def fun() -> Point:
    return Point(qubit(), 4)

@guppy
def main(p: Point) -> None:
    match fun():
        case Point(_, 4):
            measure(qubit())
        case _:
            pass

main.check()


In [5]:
# Example: linear variable used in multiple match arms (should error if not allowed)
from guppylang import guppy
from guppylang.std.quantum import qubit, measure, owned

@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def fun() -> Point:
    return Point(qubit(), 4)

@guppy
def main(p: Point @owned) -> None:
    match p:
        case Point(_, 4):
            measure(qubit())
        case _:
            pass

main.check()


Error: Drop violation (at <In[5]>:15:9)
   | 
13 | 
14 | @guppy
15 | def main(p: Point @owned) -> None:
   |          ^^^^^^^^^^^^^^^ Field `p.x` with non-droppable type `qubit` is leaked

Help: Make sure that `p.x` is consumed or returned to avoid the leak

Guppy compilation failed due to 1 previous error


In [ ]:
# Example: linear variable used in multiple match arms (should error if not allowed)
from guppylang import guppy
from guppylang.std.quantum import qubit, measure

@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def fun() -> Point:
    return Point(qubit(), 4)

@guppy
def main(p: Point) -> None:
    match p:
        case Point(_, 4):
            measure(qubit())
        case _:
            pass
    a = 1

main.check()


In [6]:
# Example: linear variable used in multiple match arms (should error if not allowed)
from guppylang import guppy
from guppylang.std.quantum import qubit, measure

@guppy.struct
class Point:
    x: qubit
    y: int

@guppy
def fun() -> Point:
    return Point(qubit(), 4)

@guppy
def describe_point(point: Point)-> None:
    pass

@guppy
def main(p: Point) -> None:
    describe_point(fun())

main.check()


Error: Drop violation (at <In[6]>:20:19)
   | 
18 | @guppy
19 | def main(p: Point) -> None:
20 |     describe_point(fun())
   |                    ^^^^^ Value with non-droppable type `Point` would be leaked after
   |                          `describe_point` returns

Help: Consider assigning the value to a local variable before passing it to
`describe_point`

Guppy compilation failed due to 1 previous error
